# AI News Agents — model-change tracker + legal subagent

This notebook demonstrates an agent that watches **AI provider websites** for model changes (launches, deprecations,
pricing, API changes) and a **legal subagent** that watches terms of service, the EU AI Act, US federal/state activity and
litigation affecting LLM providers. Both produce a cited digest saved as Markdown (emailed later).

## Design: deterministic pipeline, LLM only where judgment is needed

```
 sources ─► 1 Fetch ─► 2 Normalize ─► 3 Diff vs snapshot ─► 4 Triage (cheap LLM)
                                                              │ meaningful?
                                                              ▼
            8 Save    ◄─ 7 Digest (LLM) ◄─ 6 Dedupe ◄─ 5 Extract structured events (LLM)
                                                         + verbatim-quote guard
```

* **Detection is code, reasoning is LLM.** Fetching and diffing are deterministic and cheap; the model only sees *diffs*,
  never whole pages, and never browses on its own.
* **Every event carries a verbatim quote** from the page; events whose quote isn't in the page are dropped
  (a cheap hallucination guard).
* **The legal "subagent" is the same pipeline** with different sources, schema and prompts.
* **Seeded baselines** in `baselines/` hold older copies of each source, so the first run already reports
  real changes. A run with nothing to report writes no digest file.

## Prior art

| Category | Examples | Notes |
|---|---|---|
| Model changelog aggregators | AI Model Changelog, LLM Stats, LLM Reference, BenchLM model-changes, OpenRouter changelog, Artificial Analysis | Useful cross-checks; third-party, uneven accuracy |
| Change-detection infra | changedetection.io (OSS), PageCrawl (SaaS) | Same "detect, then reason" split |
| Terms-of-service versioning | Open Terms Archive | Versions provider ToS as git diffs |
| Legal trackers / APIs | IAPP state AI law tracker, law-firm EU AI Act trackers, Federal Register API, CourtListener, LegiScan | Primary or near-primary sources |

## 1. Setup

Auth uses Google Application Default Credentials — run once in a terminal, then set the values in the config cell below:

```
gcloud auth application-default login
```

In [ ]:
# %pip install -r requirements.txt

# All configuration lives here. Edit these values directly.

# Vertex AI
PROJECT_ID = "your-gcp-project"
REGION = "global"

# Gemini models, swappable per step.
# Check Vertex Model Garden for the model versions enabled in your project.
TRIAGE_MODEL = "gemini-2.5-flash"
EXTRACT_MODEL = "gemini-2.5-pro"
DIGEST_MODEL = "gemini-2.5-pro"

# API tokens
COURTLISTENER_TOKEN = ""  # free at courtlistener.com; leave empty to skip that source

# HTTP
HTTP_TIMEOUT_SECONDS = 30
USER_AGENT = "ai-news-agents-demo/0.1 (change monitoring)"

# Paths and limits
SNAPSHOT_DIR = "snapshots"
BASELINE_DIR = "baselines"   # seeded older copies, used when a source has no snapshot yet
OUTPUT_DIR = "output"
MAX_DIFF_CHARS = 20_000

## 2. Sources

Primary sources first. JSON APIs and RSS feeds are more stable than scraped HTML; HTML pages go through boilerplate removal.

In [ ]:
MODEL_SOURCES = [
    {"name": "Anthropic model deprecations", "provider": "Anthropic",
     "url": "https://docs.anthropic.com/en/docs/about-claude/model-deprecations"},
    {"name": "Anthropic API release notes", "provider": "Anthropic",
     "url": "https://docs.anthropic.com/en/release-notes/api"},
    {"name": "Gemini API changelog", "provider": "Google",
     "url": "https://ai.google.dev/gemini-api/docs/changelog"},
    {"name": "Vertex AI generative AI release notes", "provider": "Google",
     "url": "https://cloud.google.com/vertex-ai/generative-ai/docs/release-notes"},
    {"name": "OpenAI news (RSS)", "provider": "OpenAI", "kind": "rss",
     "url": "https://openai.com/news/rss.xml"},
    {"name": "Mistral changelog", "provider": "Mistral",
     "url": "https://docs.mistral.ai/getting-started/changelog/"},
]

LEGAL_SOURCES = [
    {"name": "Anthropic usage policy", "provider": "Anthropic",
     "url": "https://www.anthropic.com/legal/aup"},
    {"name": "OpenAI usage policies", "provider": "OpenAI",
     "url": "https://openai.com/policies/usage-policies/"},
    {"name": "EU AI Act regulatory framework", "provider": "European Commission",
     "url": "https://digital-strategy.ec.europa.eu/en/policies/regulatory-framework-ai"},
    {"name": "EU GPAI Code of Practice", "provider": "European Commission",
     "url": "https://digital-strategy.ec.europa.eu/en/policies/contents-code-gpai"},
    {"name": "Federal Register: artificial intelligence", "provider": "US Federal Register", "kind": "json",
     "url": "https://www.federalregister.gov/api/v1/documents.json"
            "?conditions[term]=%22artificial+intelligence%22&order=newest&per_page=20"
            "&fields[]=title&fields[]=type&fields[]=agencies&fields[]=publication_date&fields[]=html_url&fields[]=abstract"},
    {"name": "CourtListener: dockets naming OpenAI/Anthropic", "provider": "CourtListener", "kind": "json",
     "url": "https://www.courtlistener.com/api/rest/v4/search/"
            "?q=%22OpenAI%22+OR+%22Anthropic%22&type=r&order_by=dateFiled+desc",
     "headers": {"Authorization": f"Token {COURTLISTENER_TOKEN}"} if COURTLISTENER_TOKEN else None},
]

## 3. Event schemas

The LLM must return these shapes (structured output), so downstream code never parses free text.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field


class Triage(BaseModel):
    meaningful: bool = Field(description="False for layout, navigation, typo or boilerplate-only changes")
    category: str
    severity: Literal["low", "medium", "high"]
    reason: str


class ChangeEvent(BaseModel):
    provider: str
    model: str | None
    change_type: Literal["new_model", "deprecation", "retirement", "pricing", "api_change",
                         "rate_limit", "capability", "other"]
    effective_date: str | None = Field(description="ISO date if the page states one")
    summary: str
    quote: str = Field(description="Verbatim sentence or table row copied from the NEW page text")
    source_url: str


class LegalEvent(BaseModel):
    jurisdiction: Literal["EU", "US-federal", "US-state", "UK", "other"]
    instrument: Literal["legislation", "regulation", "guidance", "terms_of_service",
                        "litigation", "enforcement", "other"]
    title: str
    parties: list[str] = Field(description="Companies, agencies or courts involved")
    effective_date: str | None
    summary: str
    impact: str = Field(description="Practical effect on companies providing LLM services")
    quote: str = Field(description="Verbatim sentence copied from the NEW page text")
    source_url: str


class ChangeEvents(BaseModel):
    events: list[ChangeEvent]


class LegalEvents(BaseModel):
    events: list[LegalEvent]

## 4. Fetch + normalize

HTML → main text (trafilatura strips nav/footers), RSS → one line per entry, JSON → pretty-printed.

In [ ]:
import json
import feedparser
import httpx
import trafilatura

HTTP = httpx.Client(follow_redirects=True, timeout=HTTP_TIMEOUT_SECONDS, headers={"User-Agent": USER_AGENT})


def fetch(url: str, headers: dict | None = None) -> str:
    resp = HTTP.get(url, headers=headers)
    resp.raise_for_status()
    return resp.text


def normalize(raw: str, kind: str = "html") -> str:
    if kind == "rss":
        feed = feedparser.parse(raw)
        return "\n".join(f"{e.get('published', '')} | {e.get('title', '')} | {e.get('link', '')}"
                         for e in feed.entries)
    if kind == "json":
        return json.dumps(json.loads(raw), indent=1, sort_keys=True)
    text = trafilatura.extract(raw, include_tables=True, include_links=False, favor_recall=True)
    return (text or "").strip()


demo_src = MODEL_SOURCES[0]
try:
    demo_text = normalize(fetch(demo_src["url"]), demo_src.get("kind", "html"))
    print(f"{demo_src['name']}: {len(demo_text):,} chars\n")
    print(demo_text[:1500])
except httpx.HTTPError as e:
    print(f"Could not fetch {demo_src['url']}: {e}")

## 5. Snapshots + diff

Snapshots live on local disk here (GCS in production). When a source has no snapshot yet, the run diffs against its
**seeded baseline** in `baselines/<domain>/<source>.md`: the source's text with its newest entries removed, so
the first run reports real, recent changes. After that, each run diffs against the previous run's snapshot.
A source with neither a snapshot nor a baseline just saves a snapshot and reports nothing.

In [ ]:
import difflib
import hashlib
import re
from pathlib import Path


def _slug(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")


def _snapshot_path(domain: str, name: str) -> Path:
    return Path(SNAPSHOT_DIR) / domain / f"{_slug(name)}.txt"


def load_snapshot(domain: str, name: str) -> str | None:
    path = _snapshot_path(domain, name)
    return path.read_text() if path.exists() else None


def load_baseline(domain: str, name: str) -> str | None:
    path = Path(BASELINE_DIR) / domain / f"{_slug(name)}.md"
    return path.read_text() if path.exists() else None


def save_snapshot(domain: str, name: str, text: str) -> None:
    path = _snapshot_path(domain, name)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text)


def diff_text(old: str, new: str) -> str:
    # Hash first: the common case is "nothing changed" and needs no diff at all
    if hashlib.sha256(old.encode()).digest() == hashlib.sha256(new.encode()).digest():
        return ""
    lines = difflib.unified_diff(old.splitlines(), new.splitlines(), "before", "after", n=2, lineterm="")
    return "\n".join(lines)

## 6. LLM wrapper (Gemini on Vertex AI)

One function, `llm_json(prompt, schema, model)`, returns a validated pydantic object: `google-genai` with
`vertexai=True` and `response_schema`. `llm_text` returns free-form text for the digest.

In [ ]:
from functools import cache

from google.genai import types

SYSTEM = ("You analyze changes to web pages for a monitoring service. Use only facts present in the input; "
          "never infer dates, prices or model names that are not written there.")


@cache
def _gemini():
    from google import genai
    return genai.Client(vertexai=True, project=PROJECT_ID, location=REGION)


def llm_json(prompt: str, schema: type[BaseModel], model: str) -> BaseModel | None:
    resp = _gemini().models.generate_content(
        model=model, contents=prompt,
        config=types.GenerateContentConfig(system_instruction=SYSTEM,
                                           response_mime_type="application/json",
                                           response_schema=schema),
    )
    if resp.parsed is None:
        print(f"  {model} returned no parseable output; skipping")
        return None
    return resp.parsed


def llm_text(prompt: str, model: str) -> str:
    resp = _gemini().models.generate_content(
        model=model, contents=prompt,
        config=types.GenerateContentConfig(system_instruction=SYSTEM),
    )
    return resp.text

## 7. Triage + extract (with the quote guard)

Triage asks a cheap model whether a diff matters for what we track. Extraction turns the added lines into structured events, and any event whose quote isn't found in the new page text is dropped.

In [ ]:
def _clip(diff: str) -> str:
    if len(diff) <= MAX_DIFF_CHARS:
        return diff
    print(f"  diff is {len(diff):,} chars; sending the first {MAX_DIFF_CHARS:,} (raise MAX_DIFF_CHARS to send more)")
    return diff[:MAX_DIFF_CHARS]


def triage(diff: str, source: dict, focus: str) -> Triage:
    prompt = (f"Source: {source['name']} ({source['url']})\nWe track: {focus}\n\n"
              f"Unified diff of the page text:\n<diff>\n{_clip(diff)}\n</diff>\n\n"
              "Is this a meaningful change for what we track?")
    return llm_json(prompt, Triage, TRIAGE_MODEL)


def extract(diff: str, new_text: str, source: dict, domain: "Domain") -> list[BaseModel]:
    prompt = (f"Source: {source['name']} — {source['url']}\nProvider/publisher: {source['provider']}\n"
              f"We track: {domain.focus}\n\nUnified diff of the page text:\n<diff>\n{_clip(diff)}\n</diff>\n\n"
              "List one event per distinct change shown by the added (+) lines. For `quote`, copy the "
              "relevant text exactly as it appears in the + lines. Set source_url to the source URL above. "
              "Return an empty list if nothing relevant changed.")
    result = llm_json(prompt, domain.events_schema, EXTRACT_MODEL)
    events = result.events if result else []
    return [e for e in events if quote_is_verbatim(e, new_text, source)]


def _squash(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip().lower()


def quote_is_verbatim(event: BaseModel, page_text: str, source: dict) -> bool:
    event.source_url = source["url"]   # never trust the model with the citation itself
    if _squash(event.quote) in _squash(page_text):
        return True
    print(f"  dropped event (quote not found in page): {event.summary[:80]!r}")
    return False

In [ ]:
from dataclasses import dataclass
from typing import Callable


@dataclass
class Domain:
    name: str
    sources: list[dict]
    focus: str
    events_schema: type[BaseModel]
    dedupe_key: Callable[[BaseModel], tuple]
    digest_instructions: str


MODELS = Domain(
    name="models",
    sources=MODEL_SOURCES,
    focus="AI model launches, deprecations and retirement dates, pricing, API/parameter changes, rate limits, capabilities",
    events_schema=ChangeEvents,
    dedupe_key=lambda e: (e.provider.lower(), (e.model or "").lower(), e.change_type),
    digest_instructions="Group by provider. Put breaking changes (retirements, removed parameters, price increases) first.",
)

## 8. The pipeline

`run(domain)` chains fetch → normalize → diff → triage → extract → dedupe across every live source. The first run diffs against the seeded baselines; run it again and every source should report no change.

In [ ]:
import pandas as pd
from IPython.display import Markdown, display


def dedupe(events: list[BaseModel], key: Callable[[BaseModel], tuple]) -> list[BaseModel]:
    merged: dict[tuple, BaseModel] = {}
    for e in events:
        merged.setdefault(key(e), e)   # first source wins; production would merge citations
    return list(merged.values())


def run(domain: Domain) -> tuple[list[BaseModel], pd.DataFrame]:
    events, status = [], []
    for src in domain.sources:
        row = {"source": src["name"], "status": "", "events": 0}
        if "headers" in src and src["headers"] is None:
            status.append({**row, "status": "skipped: API token not set"})
            continue
        try:
            new_text = normalize(fetch(src["url"], src.get("headers")), src.get("kind", "html"))
        except httpx.HTTPStatusError as e:
            hint = " (bot-protected: needs Playwright or an Open Terms Archive feed)" if e.response.status_code == 403 else ""
            status.append({**row, "status": f"HTTP {e.response.status_code}{hint}"})
            continue
        except (httpx.HTTPError, ValueError) as e:
            status.append({**row, "status": f"fetch error: {e.__class__.__name__}"})
            continue
        if not new_text:
            status.append({**row, "status": "empty after normalize (JS-rendered? try Playwright)"})
            continue

        old_text = load_snapshot(domain.name, src["name"])
        vs = ""
        if old_text is None:
            old_text, vs = load_baseline(domain.name, src["name"]), " vs seeded baseline"
        save_snapshot(domain.name, src["name"], new_text)
        if old_text is None:
            status.append({**row, "status": "first snapshot saved"})
            continue
        diff = diff_text(old_text, new_text)
        if not diff:
            status.append({**row, "status": f"no change{vs}"})
            continue

        verdict = triage(diff, src, domain.focus)
        if not verdict or not verdict.meaningful:
            status.append({**row, "status": f"ignored{vs}: {verdict.reason if verdict else 'no verdict'}"})
            continue
        found = extract(diff, new_text, src, domain)
        events.extend(found)
        status.append({**row, "status": f"changed ({verdict.severity}){vs}", "events": len(found)})
    return dedupe(events, domain.dedupe_key), pd.DataFrame(status)


model_events, model_status = run(MODELS)
model_status

## 9. Models digest

The digest is written **only from the extracted events** (never from raw pages), and every bullet links its source. With no events the digest is empty and no LLM call is made.

In [ ]:
def digest(domain: Domain, events: list[BaseModel], title: str) -> str:
    if not events:
        return ""
    payload = json.dumps([e.model_dump() for e in events], indent=1)
    prompt = (f"Write a concise Markdown digest titled '## {title}' from these events:\n<events>\n{payload}\n</events>\n\n"
              f"{domain.digest_instructions} One bullet per event, ending with a Markdown link to its source_url. "
              "Do not add facts that are not in the events.")
    return llm_text(prompt, DIGEST_MODEL)


models_digest = digest(MODELS, model_events, "AI model changes")
display(Markdown(models_digest or "_No model changes to report._"))

## 10. Legal subagent

Same `run()` and `digest()` — only the sources, schema, focus and digest instructions differ.

In [ ]:
LEGAL = Domain(
    name="legal",
    sources=LEGAL_SOURCES,
    focus=("Legal and regulatory developments affecting companies that provide LLM services: terms of service and "
           "usage-policy changes, EU AI Act / GPAI obligations and guidance, US federal and state AI rules, "
           "enforcement actions, and lawsuits (copyright, privacy, consumer protection)"),
    events_schema=LegalEvents,
    dedupe_key=lambda e: (e.jurisdiction, _squash(e.title)),
    digest_instructions=("Group by jurisdiction, then litigation and terms-of-service changes. For each item, "
                         "state the practical impact on LLM providers. This is monitoring, not legal advice."),
)

legal_events, legal_status = run(LEGAL)
display(legal_status)
legal_digest = digest(LEGAL, legal_events, "Legal & regulatory updates for LLM providers")
display(Markdown(legal_digest or "_No legal changes to report._"))

## 11. What changed

Charts of the events the LLM extracted (after the quote guard). Colors follow the schema's fixed order for
`change_type` / `instrument`, so a category keeps its color across runs. Each chart is also saved as a PNG in
`output/` so the digest file can reference it.

In [ ]:
from datetime import date
from typing import get_args

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import MaxNLocator

SURFACE, INK, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
CHANGE_COLORS = dict(zip(get_args(ChangeEvent.model_fields["change_type"].annotation), SERIES))
INSTRUMENT_COLORS = dict(zip(get_args(LegalEvent.model_fields["instrument"].annotation), SERIES))

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "text.color": INK, "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": INK,
    "axes.edgecolor": GRID, "grid.color": GRID, "font.size": 10, "axes.titlesize": 12,
    "axes.spines.top": False, "axes.spines.right": False,
})


def _chart_path(slug: str) -> Path:
    path = Path(OUTPUT_DIR) / f"chart-{date.today().isoformat()}-{slug}.png"
    path.parent.mkdir(parents=True, exist_ok=True)
    return path


def _finish(fig, slug: str) -> Path:
    path = _chart_path(slug)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    return path


def stacked_counts(df: pd.DataFrame, row: str, stack: str, colors: dict, title: str, slug: str) -> Path | None:
    if df.empty:
        print(f"{title}: no events to chart")
        return None
    counts = pd.crosstab(df[row], df[stack])
    counts = counts[[c for c in colors if c in counts.columns]]
    counts = counts.loc[counts.sum(axis=1).sort_values().index]   # largest bar on top

    fig, ax = plt.subplots(figsize=(8, 0.45 * len(counts) + 1.6))
    left = np.zeros(len(counts))
    for col in counts.columns:
        ax.barh(counts.index, counts[col], left=left, height=0.6, color=colors[col],
                edgecolor=SURFACE, linewidth=2, label=col.replace("_", " "))
        left += counts[col].to_numpy()
    for y, total in enumerate(left):
        ax.text(total + 0.08, y, f"{int(total)}", va="center", color=MUTED)

    ax.set_title(title, loc="left")
    ax.set_xlabel("events")
    ax.set_xlim(0, left.max() * 1.08)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.xaxis.grid(True)
    ax.set_axisbelow(True)
    ax.spines["left"].set_visible(False)
    ax.tick_params(axis="y", length=0)
    if len(counts.columns) > 1:
        ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(1.04, 1), handlelength=1, handleheight=1)
    return _finish(fig, slug)


def effective_dates(model_df: pd.DataFrame, legal_df: pd.DataFrame, slug: str) -> Path | None:
    rows = []
    for domain_label, df, label in (
        ("models", model_df, lambda r: f"{r['provider']} {r['model'] or ''} · {r['change_type'].replace('_', ' ')}"),
        ("legal", legal_df, lambda r: f"{r['jurisdiction']} · {r['title'][:45]}"),
    ):
        if df.empty:
            continue
        dates = pd.to_datetime(df["effective_date"], errors="coerce")
        for (_, r), d in zip(df.iterrows(), dates):
            if pd.notna(d):
                rows.append({"domain": domain_label, "label": " ".join(label(r).split()), "date": d})
    if not rows:
        print("Effective dates: no events state a date")
        return None
    dated = pd.DataFrame(rows).sort_values("date", ascending=False).reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(8, 0.4 * len(dated) + 1.6))
    for color, domain_label in zip(SERIES, ("models", "legal")):
        part = dated[dated["domain"] == domain_label]
        if not part.empty:
            ax.scatter(part["date"], part.index, s=64, color=color, edgecolor=SURFACE, linewidth=2,
                       label=domain_label, zorder=3)
    ax.axvline(pd.Timestamp(date.today()), color=MUTED, linestyle="--", linewidth=1)
    ax.text(pd.Timestamp(date.today()), len(dated) - 0.4, " today", color=MUTED, va="bottom")
    ax.set_yticks(dated.index, dated["label"])
    ax.set_ylim(-0.6, len(dated) - 0.2)
    ax.set_title("Stated effective dates", loc="left")
    ax.xaxis.grid(True)
    ax.set_axisbelow(True)
    ax.spines["left"].set_visible(False)
    ax.tick_params(axis="y", length=0)
    fig.autofmt_xdate()
    if dated["domain"].nunique() > 1:
        ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(1.02, 1))
    return _finish(fig, slug)


model_df = pd.DataFrame([e.model_dump() for e in model_events])
legal_df = pd.DataFrame([e.model_dump() for e in legal_events])

chart_paths = [p for p in (
    stacked_counts(model_df, "provider", "change_type", CHANGE_COLORS, "Model changes by provider", "model-changes"),
    stacked_counts(legal_df, "jurisdiction", "instrument", INSTRUMENT_COLORS, "Legal updates by jurisdiction", "legal-updates"),
    effective_dates(model_df, legal_df, "effective-dates"),
) if p]

## 12. Save digest

The combined digest (with the charts above) is written to `output/digest-<date>.md` and rendered below.
If neither domain has anything to report, no file is written and `digest_path` is `None`.
Email delivery comes next.

In [ ]:
def save_digest(text: str, charts: list[Path]) -> Path:
    if charts:
        # Chart links are relative to output/, next to the digest file
        text += "\n\n## Charts\n\n" + "\n\n".join(f"![{p.stem}]({p.name})" for p in charts)
    path = Path(OUTPUT_DIR) / f"digest-{date.today().isoformat()}.md"
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text)
    print(f"saved {len(text):,} chars to {path}")
    return path


sections = [d for d in (models_digest, legal_digest) if d]
if sections:
    digest_path = save_digest("\n\n".join(sections), chart_paths)
    # Render the saved file; image links are rewritten to resolve from the notebook's directory
    display(Markdown(digest_path.read_text().replace("](chart-", f"]({OUTPUT_DIR}/chart-")))
else:
    digest_path = None
    print("No changes to report; no digest written.")

Next: email delivery — only when `digest_path` is set — e.g. via SMTP:

```python
import smtplib
from email.message import EmailMessage

msg = EmailMessage()
msg["Subject"], msg["From"], msg["To"] = "AI news digest", "bot@yourco.com", "team@yourco.com"
msg.set_content(digest_path.read_text())
with smtplib.SMTP_SSL("smtp.yourco.com") as s:
    s.login(user, password)   # from Secret Manager in production
    s.send_message(msg)
```

## 13. Productionizing

| Demo | Production |
|---|---|
| Notebook cells | `src/tracker/{fetch,normalize,diff,llm,schemas,pipeline,deliver}.py` + `tracker run --domain models\|legal` CLI |
| `output/*.md` digest file | Email (SMTP / SendGrid) |
| `snapshots/` on disk | GCS bucket (snapshots) + Firestore (events, dedupe across runs) |
| Manual run | Cloud Run Job + Cloud Scheduler (models daily, legal weekly) |
| Config cell literals | Secret Manager (SMTP, CourtListener/LegiScan tokens) |
| `httpx` only | Playwright for JS-rendered pages; `GET /v1/models` diffing per provider for new/removed model IDs |
| Eyeballing output | Golden tests: saved before/after pages → expected events; track false-positive rate |
| Single extraction pass | Cross-check against aggregators |